# Corruption Robustness Evaluation

**Corruptions:** Arc Occlusion, Gaussian Noise, Gaussian Blur

**Severities:** 0.1, 0.2, 0.4, 0.6

**Images:** Full 5,000 test set

**Metrics:** Canny IoU, Canny F1, Chamfer, Hough Circle IoU, Detection Rate, Center Error, Radius Error, CLIP Score

Progress saved to Drive after every 50 images

## Setup

In [ ]:
# install dependencies
!pip install pytorch-lightning==1.5.0 einops transformers open_clip_torch clean-fid scipy -q
print('Dependencies installed')

In [ ]:
# mount Drive
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

PROJECT_DIR = Path('/content/drive/.shortcut-targets-by-id/1aDdZyCDSiIOmlcFCcrxnXWsza4JNh6Mo/controlnet_project')
SPLIT_DIR   = PROJECT_DIR / 'splits'
CKPT_DIR    = PROJECT_DIR / 'checkpoints'

# output directories for corruption results
CORR_DIR = PROJECT_DIR / 'outputs' / 'corruption'
CORR_DIR.mkdir(parents=True, exist_ok=True)

print(f'Project directory: {PROJECT_DIR}')
print(f'Corruption output directory: {CORR_DIR}')

In [ ]:
# clone repo and download dataset
if not os.path.exists('/content/ControlNet'):
    !git clone https://github.com/lllyasviel/ControlNet.git

%cd /content/ControlNet

if not os.path.exists('./training/fill50k'):
    !wget -q https://huggingface.co/lllyasviel/ControlNet/resolve/main/training/fill50k.zip -O /tmp/fill50k.zip
    !unzip -q /tmp/fill50k.zip -d ./training/

print('Repo and dataset ready')

In [ ]:
import torch
import einops
import numpy as np
import cv2
import json
import time
import pathlib
import matplotlib.pyplot as plt
from PIL import Image
from scipy.spatial.distance import cdist
from scipy.ndimage import binary_dilation
from torch.utils.data import DataLoader, Subset, Dataset
from transformers import CLIPProcessor, CLIPModel

from tutorial_dataset import MyDataset
from cldm.model import create_model, load_state_dict
from cldm.ddim_hacked import DDIMSampler

## Corruption Functions

In [ ]:
def corrupt_arc_occlusion(edge, severity, rng=None):
    """
    Remove a contiguous arc segment from the circle contour, where severity is
    fraction of arc to remove (0.0 = no removal, 1.0 = full circle removed)
    """
    if rng is None:
        rng = np.random.default_rng()

    img_np = (edge * 255.0).astype(np.uint8)
    if img_np.ndim == 3:
        edge_gray = cv2.cvtColor(img_np, cv2.COLOR_RGB2GRAY)
    else:
        edge_gray = img_np

    ys, xs = np.where(edge_gray > 127)
    if len(xs) == 0:
        return edge

    cx, cy = xs.mean(), ys.mean()
    angles = np.arctan2(ys - cy, xs - cx)
    start  = rng.uniform(-np.pi, np.pi)
    width  = severity * 2 * np.pi
    shifted = (angles - start) % (2 * np.pi)
    remove  = shifted < width

    corrupted = img_np.copy()
    corrupted[ys[remove], xs[remove]] = 0
    return corrupted.astype(np.float32) / 255.0


def corrupt_spurious_edge_noise(edge, severity, rng=None, max_displacement=60.0):
    """
    Perturb edge pixels radially using band-limited noise (sum of harmonics),
    giving irregular organic jitter rather than a smooth regular wave.
    severity in [0,1] maps to [0, max_displacement] pixels of displacement.
    """
    if rng is None:
        rng = np.random.default_rng()

    img_np = (edge * 255.0).astype(np.uint8)
    if img_np.ndim == 3:
        edge_gray = cv2.cvtColor(img_np, cv2.COLOR_RGB2GRAY)
    else:
        edge_gray = img_np

    ys, xs = np.where(edge_gray > 127)
    if len(xs) == 0:
        return edge

    H, W = edge_gray.shape
    cx, cy = xs.mean(), ys.mean()
    angles = np.arctan2(ys - cy, xs - cx)
    order = np.argsort(angles)
    xs, ys, angles = xs[order], ys[order], angles[order]

    # More harmonics, flatter rolloff (1/sqrt(k) instead of 1/k) = more jagginess
    n_harmonics = 16
    disp = np.zeros(len(angles))
    for k in range(1, n_harmonics + 1):
        amp   = rng.uniform(0, 1) / np.sqrt(k)
        phase = rng.uniform(0, 2 * np.pi)
        disp += amp * np.sin(k * angles + phase)

    # Normalise and scale
    disp = disp / (np.max(np.abs(disp)) + 1e-8) * max_displacement * severity

    new_xs = np.clip(np.round(xs + disp * np.cos(angles)).astype(int), 0, W - 1)
    new_ys = np.clip(np.round(ys + disp * np.sin(angles)).astype(int), 0, H - 1)

    corrupted = np.zeros_like(edge_gray)
    corrupted[new_ys, new_xs] = edge_gray[ys, xs]

    if edge.ndim == 3:
        corrupted = np.stack([corrupted] * 3, axis=-1)

    return corrupted.astype(np.float32) / 255.0


def corrupt_gaussian_blur(edge, severity, rng=None):
    """
    Apply Gaussian blur to the condition image. Severity: controls kernel size (0.0 = no blur, 1.0 = heavy blur)
    """
    img_np = (edge * 255.0).astype(np.uint8)
    # Map severity to kernel size: severity 0.1->3, 0.2->7, 0.4->15, 0.6->25
    k = max(3, int(severity * 80))
    if k % 2 == 0:
        k += 1
    blurred = cv2.GaussianBlur(img_np, (k, k), 0)
    return blurred.astype(np.float32) / 255.0


CORRUPTION_FNS = {
    'arc_occlusion':  corrupt_arc_occlusion,
    'spurious_edge_noise': corrupt_spurious_edge_noise,
    'gaussian_blur':  corrupt_gaussian_blur,
}

print('Corruption functions defined')

In [ ]:
class CorruptedDataset(Dataset):
    def __init__(self, base_dataset, corruption, severity):
        self.base_dataset = base_dataset
        self.corruption   = corruption
        self.severity     = severity
        self.corrupt_fn   = CORRUPTION_FNS[corruption]

    def __len__(self):
        return len(self.base_dataset)

    def __getitem__(self, idx):
        item   = self.base_dataset[idx]
        rng    = np.random.default_rng(seed=idx)  # reproducible per-image
        hint   = self.corrupt_fn(item['hint'], self.severity, rng=rng)
        return {'jpg': item['jpg'], 'hint': hint, 'txt': item['txt']}

print('CorruptedDataset defined')

## Visualize Corruptions

In [ ]:
# sanity check of each corruption at each severity
import torch
from torch.utils.data import Subset

split   = torch.load(str(SPLIT_DIR / 'split_seed42.pt'))
dataset = MyDataset()
test_ds = Subset(dataset, split['test'])

item      = test_ds[0]
condition = item['hint']  # [0,1] HWC

corruptions = ['arc_occlusion', 'spurious_edge_noise', 'gaussian_blur']
severities  = [0.1, 0.2, 0.4, 0.6]

fig, axes = plt.subplots(len(corruptions), len(severities) + 1,
                          figsize=(14, 9))

for row, corruption in enumerate(corruptions):
    axes[row, 0].imshow(condition)
    axes[row, 0].set_title('Clean', fontsize=9)
    axes[row, 0].set_ylabel(corruption, fontsize=9, fontweight='bold')
    axes[row, 0].axis('off')

    corrupt_fn = CORRUPTION_FNS[corruption]
    for col, severity in enumerate(severities):
        corrupted = corrupt_fn(condition, severity, rng=np.random.default_rng(0))
        axes[row, col+1].imshow(corrupted)
        axes[row, col+1].set_title(f'severity={severity}', fontsize=9)
        axes[row, col+1].axis('off')

plt.suptitle('Corruption Visualizations', fontsize=12)
plt.tight_layout()
plt.savefig(str(CORR_DIR / 'corruption_visualization.png'), dpi=120, bbox_inches='tight')
plt.show()
print('Saved corruption visualization')

## Load model, define metrics functions

In [ ]:
# Load final checkpoint
import pathlib
torch.serialization.add_safe_globals([pathlib.PosixPath])

CKPT_PATH = str(CKPT_DIR / 'controlnet-final.ckpt')
print(f'Loading: {CKPT_PATH}')

model = create_model('./models/cldm_v15.yaml').cuda()
model.load_state_dict(load_state_dict(CKPT_PATH, location='cuda'))
model.eval()
sampler = DDIMSampler(model)

print('Model loaded')

In [ ]:
def run_inference(condition, prompt, ddim_steps=20, scale=9.0):
    H, W, C = condition.shape
    control = torch.from_numpy(condition.copy()).float().cuda().unsqueeze(0)
    control = einops.rearrange(control, 'b h w c -> b c h w').clone()

    cond = {
        'c_concat':    [control],
        'c_crossattn': [model.get_learned_conditioning([prompt])]
    }
    un_cond = {
        'c_concat':    [control],
        'c_crossattn': [model.get_learned_conditioning([''])]
    }
    shape = (4, H // 8, W // 8)

    with torch.no_grad():
        samples, _ = sampler.sample(
            S=ddim_steps,
            conditioning=cond,
            batch_size=1,
            shape=shape,
            verbose=False,
            unconditional_guidance_scale=scale,
            unconditional_conditioning=un_cond,
            eta=0.0,
        )
        x_samples = model.decode_first_stage(samples)
        x_samples = einops.rearrange(x_samples, 'b c h w -> b h w c') * 127.5 + 127.5
        generated = x_samples.cpu().numpy().clip(0, 255).astype(np.uint8)[0]

    return generated

print('Inference function defined')

In [ ]:
# Metric functions (copied from eval notebook)
def binarize_condition(condition):
    if len(condition.shape) == 3:
        gray = condition.mean(axis=2)
    else:
        gray = condition
    return gray > 0.5 if gray.max() <= 1.0 else gray > 127

def extract_edges_canny(img, low=50, high=150):
    gray  = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    return cv2.Canny(gray, low, high)

def canny_iou(generated_img, original_condition):
    extracted = extract_edges_canny(generated_img) > 0
    orig      = binarize_condition(original_condition)
    intersection = np.logical_and(extracted, orig).sum()
    union        = np.logical_or(extracted, orig).sum()
    return float(intersection / (union + 1e-8))

def canny_f1(generated_img, original_condition, tolerance=3):
    extracted    = extract_edges_canny(generated_img) > 0
    orig         = binarize_condition(original_condition)
    gt_dilated   = binary_dilation(orig,      iterations=tolerance)
    pred_dilated = binary_dilation(extracted, iterations=tolerance)
    tp = np.logical_and(extracted, gt_dilated).sum()
    fp = np.logical_and(extracted, ~gt_dilated).sum()
    fn = np.logical_and(orig,      ~pred_dilated).sum()
    precision = tp / (tp + fp + 1e-8)
    recall    = tp / (tp + fn + 1e-8)
    return float(2 * precision * recall / (precision + recall + 1e-8))

def chamfer_distance(generated_img, original_condition):
    extracted = extract_edges_canny(generated_img)
    orig      = binarize_condition(original_condition)
    pts_pred  = np.argwhere(extracted > 0)
    pts_gt    = np.argwhere(orig)
    if len(pts_pred) == 0 or len(pts_gt) == 0:
        return float('inf')
    D = cdist(pts_pred, pts_gt)
    return float((D.min(axis=1).mean() + D.min(axis=0).mean()) / 2)

def fit_circle_from_3pts(p1, p2, p3):
    x1,y1=p1; x2,y2=p2; x3,y3=p3
    A = np.array([[2*(x2-x1),2*(y2-y1)],[2*(x3-x1),2*(y3-y1)]])
    b = np.array([x2**2+y2**2-x1**2-y1**2, x3**2+y3**2-x1**2-y1**2])
    if abs(np.linalg.det(A)) < 1e-6:
        return None
    cx, cy = np.linalg.solve(A, b)
    return cx, cy, np.sqrt((x1-cx)**2+(y1-cy)**2)

def ransac_circle(edge, n_iters=500, threshold=2.0, min_inliers=100):
    """
    Fit a circle robustly from edge pixels using RANSAC.
    Works on partial/cut-off circles where Hough fails.
    Returns (cx, cy, r) or None.
    """
    if edge.ndim == 3:
        edge = cv2.cvtColor(edge, cv2.COLOR_RGB2GRAY)
    ys, xs = np.where(edge > 0)
    pts = np.stack([xs, ys], axis=1)
    if len(pts) < 3:
        return None

    best_circle  = None
    best_inliers = 0
    rng = np.random.default_rng(0)

    for _ in range(n_iters):
        sample = pts[rng.choice(len(pts), 3, replace=False)]
        circle = fit_circle_from_3pts(sample[0], sample[1], sample[2])

        if circle is None:
            continue

        cx, cy, r = circle
        dists    = np.sqrt((pts[:, 0] - cx)**2 + (pts[:, 1] - cy)**2)
        inliers  = np.abs(dists - r) < threshold
        n_inliers = inliers.sum()

        if n_inliers > best_inliers:
            best_inliers = n_inliers
            best_circle  = circle

    if best_circle is None or best_inliers < min_inliers:
        return None

    return best_circle  # (cx, cy, r)

def ransac_circle_generated(edge, orig_circle, n_iters=500, threshold=2.0, min_inliers=100):
    """
    Fit a circle from generated image edge pixels using RANSAC,
    constrained by the known condition circle to reject false positives.
    Returns (cx, cy, r) or None.
    """
    if edge.ndim == 3:
        edge = cv2.cvtColor(edge, cv2.COLOR_RGB2GRAY)
    ys, xs = np.where(edge > 0)
    pts = np.stack([xs, ys], axis=1)
    if len(pts) < 3:
        return None

    orig_cx, orig_cy, orig_r = orig_circle

    best_circle  = None
    best_inliers = 0
    rng = np.random.default_rng(0)

    for _ in range(n_iters):
        sample = pts[rng.choice(len(pts), 3, replace=False)]
        circle = fit_circle_from_3pts(sample[0], sample[1], sample[2])

        if circle is None:
            continue

        cx, cy, r = circle

        # reject if radius is implausible relative to condition
        if abs(r - orig_r) / orig_r > 0.5:
            continue

        # reject if center is too far from condition center
        if np.sqrt((cx - orig_cx)**2 + (cy - orig_cy)**2) / orig_r > 0.5:
            continue

        dists     = np.sqrt((pts[:, 0] - cx)**2 + (pts[:, 1] - cy)**2)
        inliers   = np.abs(dists - r) < threshold
        n_inliers = inliers.sum()

        if n_inliers > best_inliers:
            best_inliers = n_inliers
            best_circle  = circle

    if best_circle is None or best_inliers < min_inliers:
        return None

    return best_circle

def extract_circle_ransac(img, orig_circle):
    edges = extract_edges_canny(img)
    result = ransac_circle_generated(edges, orig_circle)
    if result is not None:
          return np.array(result)

def ransac_metrics(generated_img, original_condition):
    orig_circle = ransac_circle(binarize_condition(original_condition).astype(np.uint8) * 255)
    gen_circle = extract_circle_ransac(generated_img, orig_circle)

    if orig_circle is None or gen_circle is None:
        return {
            'detected':                False,
            'center_error':            None,
            'radius_error':            None,
            'normalized_center_error': None,
            'normalized_radius_error': None,
            'circle_iou':              None,
        }

    orig_x, orig_y, orig_r = orig_circle
    gen_x,  gen_y,  gen_r  = gen_circle

    center_error = float(np.sqrt((orig_x - gen_x)**2 + (orig_y - gen_y)**2))
    radius_error = float(abs(orig_r - gen_r))

    H, W = original_condition.shape[:2]
    mask_orig = np.zeros((H, W), dtype=np.uint8)
    mask_gen  = np.zeros((H, W), dtype=np.uint8)
    cv2.circle(mask_orig, (int(orig_x), int(orig_y)), int(orig_r), 255, -1)
    cv2.circle(mask_gen,  (int(gen_x),  int(gen_y)),  int(gen_r),  255, -1)

    intersection = np.logical_and(mask_orig, mask_gen).sum()
    union        = np.logical_or(mask_orig,  mask_gen).sum()
    circle_iou   = float(intersection / (union + 1e-8))

    return {
        'detected':                True,
        'center_error':            center_error,
        'radius_error':            radius_error,
        'normalized_center_error': center_error / orig_r,
        'normalized_radius_error': radius_error / orig_r,
        'circle_iou':              circle_iou,
    }

# CLIP
print('Loading CLIP...')
clip_model     = CLIPModel.from_pretrained('openai/clip-vit-base-patch32').cuda()
clip_processor = CLIPProcessor.from_pretrained('openai/clip-vit-base-patch32')
clip_model.eval()

def compute_clip_score(generated_img, prompt):
    pil_img = Image.fromarray(generated_img)
    inputs  = clip_processor(text=[prompt], images=pil_img,
                              return_tensors='pt', padding=True)
    inputs  = {k: v.cuda() for k, v in inputs.items()}
    with torch.no_grad():
        outputs = clip_model(**inputs)
    return float(outputs.logits_per_image.item())

class NumpyEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, (np.floating,)): return float(obj)
        if isinstance(obj, (np.integer,)):  return int(obj)
        if isinstance(obj, np.ndarray):     return obj.tolist()
        return super().default(obj)

print('All metric functions defined')

In [ ]:
!pip install lpips -q
import lpips

In [ ]:
import lpips
loss_fn_lpips = lpips.LPIPS(net='alex').cuda()

def compute_lpips(generated_img, ground_truth_img):
    """ground_truth_img is HWC uint8 [0,255]"""
    gen = torch.from_numpy(
        generated_img.astype(np.float32) / 127.5 - 1
    ).permute(2,0,1).unsqueeze(0).cuda()

    gt = torch.from_numpy(
        ground_truth_img.astype(np.float32) / 127.5 - 1
    ).permute(2,0,1).unsqueeze(0).cuda()

    with torch.no_grad():
        score = loss_fn_lpips(gen, gt).item()
    return float(score)

## Corruption Evaluation Loop

Runs all 3 corruptions × 4 severities = 12 conditions × 300 images. Progress is saved to Drive every 50 images per condition.

In [ ]:
# Use 300 image subset for corruption study
CORRUPTION_TEST_SIZE = 300
corruption_test_ds = torch.utils.data.Subset(test_ds, range(CORRUPTION_TEST_SIZE))

In [ ]:
# import os
# prog_file = CORR_DIR / 'arc_occlusion_s0.1' / 'progress.json'
# os.remove(prog_file)

In [ ]:
CORRUPTIONS = ['arc_occlusion', 'spurious_edge_noise', 'gaussian_blur']
SEVERITIES  = [0.0, 0.1, 0.2, 0.4, 0.6]

for corruption in CORRUPTIONS:
    for severity in SEVERITIES:

        run_key  = f'{corruption}_s{severity}'
        run_dir  = CORR_DIR / run_key
        run_dir.mkdir(parents=True, exist_ok=True)
        prog_file = run_dir / 'progress.json'

        # Load existing progress
        if prog_file.exists():
            with open(prog_file) as f:
                progress = json.load(f)
            done_idxs   = set(progress['done_idxs'])
            all_results = progress['results']
            print(f'[{run_key}] Resuming from {len(done_idxs)}/{len(corruption_test_ds)}')
        else:
            done_idxs   = set()
            all_results = []
            print(f'[{run_key}] Starting fresh')

        if len(done_idxs) >= len(corruption_test_ds):
            print(f'[{run_key}] Already complete, skipping')
            continue

        # Build corrupted dataset
        corrupted_ds = CorruptedDataset(corruption_test_ds, corruption, severity)

        t_start = time.time()

        for idx in range(len(corruption_test_ds)):
            if idx in done_idxs:
                continue
            print("idx:", idx)
            item              = corrupted_ds[idx]
            condition         = item['hint']   # corrupted condition
            clean_condition   = corruption_test_ds[idx]['hint']  # original clean condition
            prompt            = item['txt']
            ground_truth      = ((corruption_test_ds[idx]['jpg'] + 1.0) * 127.5).clip(0, 255).astype(np.uint8)


            # Generate
            generated = run_inference(condition, prompt)

            # Metrics, compare generated against CLEAN condition
            h = ransac_metrics(generated, clean_condition)
            print(h)

            result = {
                'idx':               idx,
                'corruption':        corruption,
                'severity':          severity,
                'clip_score':        compute_clip_score(generated, prompt),
                'canny_iou':         canny_iou(generated, clean_condition),
                'canny_f1':          canny_f1(generated, clean_condition),
                'chamfer':           chamfer_distance(generated, clean_condition),
                'detected':          h['detected'],
                'circle_iou':        h['circle_iou'],
                'center_error':      h['center_error'],
                'radius_error':      h['radius_error'],
                'norm_center_error': h['normalized_center_error'],
                'norm_radius_error': h['normalized_radius_error'],
                'lpips': compute_lpips(generated, ground_truth),
            }

            all_results.append(result)
            done_idxs.add(idx)

            # Progress reporting
            if idx % 50 == 0:
              elapsed = time.time() - t_start
              remaining = len(done_idxs)
              rate = remaining / elapsed if elapsed > 0 else 1
              eta_s = (len(corruption_test_ds) - len(done_idxs)) / rate
              eta_min = eta_s / 60
              with open(prog_file, 'w') as f:
                  json.dump({'done_idxs': list(done_idxs),
                            'results':   all_results}, f, cls=NumpyEncoder)
              print(f'  [{run_key}] {len(done_idxs)}/{len(corruption_test_ds)} '
                    f'| rate={rate:.1f} img/s '
                    f'| ETA={eta_min:.0f} min')
        # Final save
        with open(prog_file, 'w') as f:
            json.dump({'done_idxs': list(done_idxs),
                       'results':   all_results}, f, cls=NumpyEncoder)
        with open(run_dir / 'results_final.json', 'w') as f:
            json.dump(all_results, f, cls=NumpyEncoder)

        print(f'[{run_key}] COMPLETE — {len(all_results)} images')

print('\nAll corruption evaluations complete!')

## Summarize/plot results

In [ ]:
import pandas as pd

CORRUPTIONS = ['arc_occlusion', 'spurious_edge_noise', 'gaussian_blur']
SEVERITIES  = [0.0, 0.1, 0.2, 0.4, 0.6]
summary_df = pd.read_csv(str(CORR_DIR / 'corruption_summary.csv'))

In [ ]:
# Load all results and compute summary statistics
summary = []

for corruption in CORRUPTIONS:
    for severity in SEVERITIES:
        run_key   = f'{corruption}_s{severity}'
        res_file  = CORR_DIR / run_key / 'results_final.json'

        if not res_file.exists():
            print(f'Missing: {run_key}')
            continue

        with open(res_file) as f:
            results = json.load(f)

        df = pd.DataFrame(results)

        def mean_notnan(col):
            vals = df[col].dropna()
            vals = vals[vals != float('inf')]
            return float(vals.mean()) if len(vals) > 0 else None

        summary.append({
            'corruption':       corruption,
            'severity':         severity,
            'n':                len(results),
            'clip_score':       mean_notnan('clip_score'),
            'lpips': mean_notnan('lpips'),
            'canny_iou':        mean_notnan('canny_iou'),
            'canny_f1':         mean_notnan('canny_f1'),
            'chamfer':          mean_notnan('chamfer'),
            'detection_rate':   float(df['detected'].mean()),
            'circle_iou':       mean_notnan('circle_iou'),
            'center_error':     mean_notnan('center_error'),
            'radius_error':     mean_notnan('radius_error'),
        })

summary_df = pd.DataFrame(summary)
print(summary_df.to_string())

# Save summary
summary_df.to_csv(str(CORR_DIR / 'corruption_summary.csv'), index=False)
print('\nSaved summary CSV')

In [ ]:
# Plot metric vs severity curves for each corruption type
CORRUPTION_LABELS = {
    'arc_occlusion':  'Arc Occlusion',
    'spurious_edge_noise': 'Edge Noise',
    'gaussian_blur':  'Gaussian Blur',
}
COLORS = {
    'arc_occlusion':  '#e41a1c',
    'spurious_edge_noise': '#377eb8',
    'gaussian_blur':  '#4daf4a',
}

# Clean baseline values (from clean eval)
CLEAN_BASELINE = {
    'canny_iou':      0.156,
    'canny_f1':       0.620,
    'circle_iou':     0.910,
    'detection_rate': 0.603,
    'clip_score':     30.26,
    'lpips': 0.5237,
}

metrics_to_plot = [
    ('canny_iou',      'Canny IoU',        'higher=better'),
    ('canny_f1',       'Canny F1',         'higher=better'),
    ('circle_iou',     'Circle IoU',       'higher=better'),
    ('detection_rate', 'Detection Rate',   'higher=better'),
    ('clip_score',     'CLIP Score',       'higher=better'),
    ('chamfer',        'Chamfer Distance', 'lower=better'),
    ('lpips',          'LPIPS',            'lower=better'),
]

fig, axes = plt.subplots(2, 3, figsize=(14, 8))

for ax, (metric, label, direction) in zip(axes.flat, metrics_to_plot):
    for corruption in CORRUPTIONS:
        sub = summary_df[summary_df['corruption'] == corruption].sort_values('severity')
        if len(sub) == 0:
            continue

        baseline = sub[sub['severity'] == 0.0][metric].values
        if len(baseline) == 0 or baseline[0] == 0:
            continue
        normalized = sub[metric] / baseline[0]

        ax.plot(sub['severity'], normalized,
                marker='o', color=COLORS[corruption],
                label=CORRUPTION_LABELS[corruption], linewidth=2)

    ax.axhline(1.0, color='black', linestyle='--', linewidth=1.5,
               label='CS=0 baseline', zorder=1)
    ax.set_xlabel('Severity', fontsize=10)
    ax.set_ylabel(f'{label} (relative to CS=0)', fontsize=10)
    ax.set_title(f'{label}\n({direction})', fontsize=10)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(str(CORR_DIR / 'corruption_curves.png'), dpi=150, bbox_inches='tight')
plt.savefig(str(CORR_DIR / 'corruption_curves.pdf'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved corruption curves')

In [ ]:
import seaborn as sns
sns.set_style("whitegrid")
# Plot metric degradation per corruption type
CORRUPTION_LABELS = {
    'arc_occlusion':       'Arc Occlusion',
    'spurious_edge_noise': 'Edge Noise',
    'gaussian_blur':       'Gaussian Blur',
}

metrics_to_plot = [
    # ('canny_iou',      'Canny IoU',        'higher=better'),
    ('canny_f1',       'Canny F1',         'higher=better'),
    ('circle_iou',     'Circle IoU',       'higher=better'),
    ('detection_rate', 'Detection Rate',   'higher=better'),
    ('clip_score',     'CLIP Score',       'higher=better'),
    ('chamfer',        'Chamfer Distance', 'lower=better'),
    ('lpips',          'LPIPS',            'lower=better'),
]

fig, axes = plt.subplots(1, len(CORRUPTIONS), figsize=(15, 5))
fig.suptitle('Metric Degradation by Corruption Type', fontsize=12)

for ax, corruption in zip(axes, CORRUPTIONS):
    for metric, label, direction in metrics_to_plot:
        sub = summary_df[summary_df['corruption'] == corruption].sort_values('severity')
        if len(sub) == 0:
            continue

        baseline = sub[sub['severity'] == 0.0][metric].values
        if len(baseline) == 0 or baseline[0] == 0:
            continue

        normalized = sub[metric] / baseline[0]

        # flip lower=better metrics so down always means worse
        if direction == 'lower=better':
            normalized = 1 / normalized

        ax.plot(sub['severity'], normalized,
                marker='o', label=label, linewidth=2)

    ax.axhline(1.0, color='black', linestyle='--', linewidth=1.5,
               label='CS=0 baseline', zorder=1)
    ax.set_title(CORRUPTION_LABELS[corruption], fontsize=11)
    # ax.set_xlabel('Severity', fontsize=10)
    # ax.set_ylabel('Relative performance\n(down = worse)', fontsize=10)
    ax.set_ylim(0, 1.2)
    ax.set_xticklabels([])
    ax.set_yticklabels([])

    # ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(str(CORR_DIR / 'corruption_per_type.png'), dpi=150, bbox_inches='tight')
plt.savefig(str(CORR_DIR / 'corruption_per_type.pdf'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved corruption per type curves')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

CORRUPTIONS = ['arc_occlusion', 'spurious_edge_noise', 'gaussian_blur']
SEVERITIES  = [0.0, 0.1, 0.2, 0.4, 0.6]
EXAMPLE_IDXS = [0, 1, 2]

for example_idx in EXAMPLE_IDXS:
    item   = corruption_test_ds[example_idx]
    clean  = item['hint']
    prompt = item['txt']

    n_rows = len(CORRUPTIONS)
    n_cols = 1 + len(SEVERITIES) * 2

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 2.5, n_rows * 2.5))
    fig.suptitle(f'Example {example_idx} — "{prompt}"', fontsize=10, y=1.01)

    col_titles = ['Clean']
    for s in SEVERITIES:
        col_titles += [f'Corrupt\nCS={s}', f'Generated\nCS={s}']
    for j, title in enumerate(col_titles):
        axes[0, j].set_title(title, fontsize=7)

    for row, corruption in enumerate(CORRUPTIONS):
        axes[row, 0].set_ylabel(corruption.replace('_', '\n'), fontsize=7,
                                rotation=0, labelpad=55, va='center')
        axes[row, 0].imshow((clean * 255).astype(np.uint8))
        axes[row, 0].axis('off')

        for col_offset, severity in enumerate(SEVERITIES):
            corrupted_ds        = CorruptedDataset(corruption_test_ds, corruption, severity)
            corrupted_condition = corrupted_ds[example_idx]['hint']
            generated           = run_inference(corrupted_condition, prompt)

            col_cond = 1 + col_offset * 2
            col_gen  = col_cond + 1

            axes[row, col_cond].imshow((corrupted_condition * 255).astype(np.uint8))
            axes[row, col_cond].axis('off')
            axes[row, col_gen].imshow(generated)
            axes[row, col_gen].axis('off')

    plt.tight_layout()
    plt.show()